In [ ]:
import os
import sys

import collections
import xarray as xr
xr.set_options(keep_attrs=True)
import netCDF4 as nc
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
import metpy.calc as mp
from metpy.units import units
from scipy.stats import ttest_ind, ttest_rel
from datetime import datetime

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

In [ ]:
### +++ LOAD CLIMATE DATA +++ ###

diags=['h200','h700']
dat={ 'merra2':{}, 'ctrl':{}, 'max.global':{} }

for key in dat.keys():
    for varn in diags:
        if key=='merra2':
            dat[key][varn] = xr.open_dataset(f'obs.merra2.{varn}.nc')[varn]
        else:
            dat[key][varn] = xr.open_dataset(f'aogcm.{key}.{varn}.nc')[varn]
        
## calculate annual mean fields for sst and streamfunction
# number of years to omit 
omit_yrs = 30
n_omit = omit_yrs * 12
tmax = 350 * 12
sst = {}
sf_Atl = {}
for key in dat.keys(): 
    sst[key] = dat[key]['sst'].isel(time=slice(n_omit,tmax)).mean(dim='time')
    sf_Atl[key] = dat[key]['sf_Atl'].isel(time=slice(n_omit,tmax)).mean(dim='time').where(ctrl > -6.8550344e+20, np.nan)